<a href="https://colab.research.google.com/github/Bukunmi2108/ml-journey/blob/main/research/p2/lora.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

LoRA from Scratch

In [38]:
import math
import torch
import torch.nn as nn

Hyper Parameters

In [39]:
rank = 8
alpha = 2 * rank
dropout = 0.05

LoRA Linear Class

In [40]:
class LoRALinear(nn.Module):
    def __init__(self, base: nn.Linear, rank: int, alpha: int, dropout: float = dropout):
        super().__init__()
        self.base = base
        self.rank = rank
        self.alpha = alpha
        self.scaling = alpha / rank
        self.dropout = nn.Dropout(dropout)

        for p in self.base.parameters():
            p.requires_grad = False

        self.A = nn.Linear(base.in_features, rank, bias=False)
        self.B = nn.Linear(rank, base.out_features, bias=False)

        nn.init.kaiming_uniform_(self.A.weight, a=math.sqrt(5))
        nn.init.zeros_(self.B.weight)

    def forward(self, x):
        return self.base(x) + self.B(self.A(self.dropout(x))) * self.scaling

Replace LoRA Modules

In [41]:
def replace_lora_modules(model: nn.Module, target_names: tuple, rank: int, alpha: int):
    for name, module in list(model.named_modules()):
        if any(name.endswith(t) for t in target_names) and isinstance(module, nn.Linear):
            if '.' not in name:
                parent = model
                child_name = name
            else:
                parent_name, child_name = name.rsplit(".", 1)
                parent = model.get_submodule(parent_name)
            setattr(parent, child_name, LoRALinear(module, rank, alpha))

A simple model to test

In [42]:
class SimpleModel(nn.Module):
  def __init__(self, in_features: int, out_features: int):
    super().__init__()
    self.w1 = nn.Linear(in_features, 2 * in_features)
    self.w2 = nn.Linear(2 * in_features, out_features)
    self.relu = nn.ReLU()
  def forward(self, x: torch.Tensor):
    return self.w2(self.relu(self.w1(x)))

Testing `replace_lora_modules` with `SimpleModel`

In [43]:
in_features = 10
out_features = 5
model = SimpleModel(in_features, out_features)

print("Original SimpleModel Architecture:")
print(model)

Original SimpleModel Architecture:
SimpleModel(
  (w1): Linear(in_features=10, out_features=20, bias=True)
  (w2): Linear(in_features=20, out_features=5, bias=True)
  (relu): ReLU()
)


In [44]:
target_names = ("w1", "w2")

replace_lora_modules(model, target_names, rank, alpha)

print("\nSimpleModel Architecture after applying LoRA:")
print(model)


SimpleModel Architecture after applying LoRA:
SimpleModel(
  (w1): LoRALinear(
    (base): Linear(in_features=10, out_features=20, bias=True)
    (dropout): Dropout(p=0.05, inplace=False)
    (A): Linear(in_features=10, out_features=8, bias=False)
    (B): Linear(in_features=8, out_features=20, bias=False)
  )
  (w2): LoRALinear(
    (base): Linear(in_features=20, out_features=5, bias=True)
    (dropout): Dropout(p=0.05, inplace=False)
    (A): Linear(in_features=20, out_features=8, bias=False)
    (B): Linear(in_features=8, out_features=5, bias=False)
  )
  (relu): ReLU()
)


In [45]:
# Verify that the layers have been replaced with LoRALinear
assert isinstance(model.w1, LoRALinear), "w1 was not replaced by LoRALinear"
assert isinstance(model.w2, LoRALinear), "w2 was not replaced by LoRALinear"
print("\nVerification successful: w1 and w2 are now LoRALinear instances.")


Verification successful: w1 and w2 are now LoRALinear instances.


Loading and applying LoRA on an actual model

In [46]:
from transformers import AutoTokenizer, AutoModelForCausalLM

In [47]:
model_id = "Qwen/Qwen2.5-0.5B-Instruct"
tokenizer = AutoTokenizer.from_pretrained(model_id)
model = AutoModelForCausalLM.from_pretrained(
    model_id,
    torch_dtype="auto",
    device_map="auto"
)

Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

In [48]:
# Freezing model's params

for p in model.parameters():
  p.requires_grad = False

In [49]:
targets = ("q_proj", "v_proj")
replace_lora_modules(model, targets, rank, alpha)

In [50]:
model

Qwen2ForCausalLM(
  (model): Qwen2Model(
    (embed_tokens): Embedding(151936, 896)
    (layers): ModuleList(
      (0-23): 24 x Qwen2DecoderLayer(
        (self_attn): Qwen2Attention(
          (q_proj): LoRALinear(
            (base): Linear(in_features=896, out_features=896, bias=True)
            (dropout): Dropout(p=0.05, inplace=False)
            (A): Linear(in_features=896, out_features=8, bias=False)
            (B): Linear(in_features=8, out_features=896, bias=False)
          )
          (k_proj): Linear(in_features=896, out_features=128, bias=True)
          (v_proj): LoRALinear(
            (base): Linear(in_features=896, out_features=128, bias=True)
            (dropout): Dropout(p=0.05, inplace=False)
            (A): Linear(in_features=896, out_features=8, bias=False)
            (B): Linear(in_features=8, out_features=128, bias=False)
          )
          (o_proj): Linear(in_features=896, out_features=896, bias=False)
        )
        (mlp): Qwen2MLP(
          (gate

In [51]:
trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
total = sum(p.numel() for p in model.parameters())
print(f"Trainable parameters: {trainable}")
print(f"Total parameters: {total}")
print(f"Percentage of trainable parameters: {trainable / total * 100:.2f}%")

Trainable parameters: 540672
Total parameters: 494573440
Percentage of trainable parameters: 0.11%


Load Dataset

In [52]:
from datasets import load_dataset

ds = load_dataset("HuggingFaceH4/CodeAlpaca_20K")

In [53]:
ds

DatasetDict({
    train: Dataset({
        features: ['prompt', 'completion'],
        num_rows: 18019
    })
    test: Dataset({
        features: ['prompt', 'completion'],
        num_rows: 2003
    })
})

In [54]:
def format_example(example):
  prompt_text = example["prompt"]
  completion_text = example["completion"]
  return {"text": f"{prompt_text}\n{completion_text}"}

In [55]:
formatted_ds = ds.map(format_example, remove_columns=['prompt', 'completion'], num_proc=4)

print("\nFormatted dataset structure:")
print(formatted_ds)
print("\nExample of a formatted prompt:")
print(formatted_ds['train'][0]['text'])


Formatted dataset structure:
DatasetDict({
    train: Dataset({
        features: ['text'],
        num_rows: 18019
    })
    test: Dataset({
        features: ['text'],
        num_rows: 2003
    })
})

Example of a formatted prompt:
Create a Java class which sorts the given array of numbers.
[9, 2, 4, 3, 6, 1]
class ArraySort { 
  
    void sort(int arr[]) { 
        int n = arr.length; 
  
        // One by one move boundary of unsorted subarray 
        for (int i = 0; i < n-1; i++) { 
            
            // Find the minimum element in unsorted array 
            int min_index = i; 
            for (int j = i+1; j < n; j++) 
                if (arr[j] < arr[min_index]) 
                    min_index = j; 
  
            // Swap the found minimum element with the first element 
            int temp = arr[min_index]; 
            arr[min_index] = arr[i]; 
            arr[i] = temp; 
        } 
    } 
  
    // Prints the array 
    void printArray(int arr[]) { 
        int n =

Tokenize Dataset

In [56]:
def tokenize_function(examples):
  return tokenizer(examples['text'], truncation=True, padding=False, max_length=512)
tokenized_ds = formatted_ds.map(tokenize_function, batched=True, remove_columns=["text"], num_proc=4)

In [57]:
tokenizer.pad_token

'<|endoftext|>'

In [58]:
from transformers import DataCollatorForLanguageModeling

data_collator = DataCollatorForLanguageModeling(tokenizer=tokenizer, mlm=False)

In [62]:
from transformers import TrainingArguments, Trainer

output_dir = "./qwen_codealpaca_lora_finetuned"

training_args = TrainingArguments(
    output_dir=output_dir,
    num_train_epochs=3,
    per_device_train_batch_size=4,
    per_device_eval_batch_size=4,
    gradient_accumulation_steps=8,
    save_strategy="epoch",
    logging_dir=f"{output_dir}/logs",
    logging_strategy="steps",
    logging_steps=50,
    learning_rate=2e-4,
    weight_decay=0.01,
    fp16=True,
    push_to_hub=False,
    report_to="none",
    remove_unused_columns=False,
    label_names=["labels"]
)


[transformers] `logging_dir` is deprecated and will be removed in v5.2. Please set `TENSORBOARD_LOGGING_DIR` instead.


In [63]:
trainer = Trainer(model=model, args=training_args, data_collator=data_collator, train_dataset=tokenized_ds["train"], eval_dataset=tokenized_ds["test"])

In [64]:
trainer.train()

Step,Training Loss
50,0.925390
100,0.879648
150,0.886443
200,0.867782
250,0.871273
300,0.853916
350,0.862114
400,0.864374
450,0.858088
500,0.859212


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

TrainOutput(global_step=1692, training_loss=0.8476550776344101, metrics={'train_runtime': 2498.2597, 'train_samples_per_second': 21.638, 'train_steps_per_second': 0.677, 'total_flos': 1.6200008601072384e+16, 'train_loss': 0.8476550776344101, 'epoch': 3.0})

In [65]:
lora_output_dir = f"{output_dir}/lora_adapters"
trainer.save_model(lora_output_dir)
print(f"LoRA adapter weights saved to: {lora_output_dir}")

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

LoRA adapter weights saved to: ./qwen_codealpaca_lora_finetuned/lora_adapters


Load Fine-tuned model

In [70]:
model_for_inference = AutoModelForCausalLM.from_pretrained(
    model_id,
    torch_dtype="auto",
    device_map="auto"
)

replace_lora_modules(model_for_inference, targets, rank, alpha)

try:
    from safetensors.torch import load_file
    saved_model_path = f"{lora_output_dir}/model.safetensors"
    saved_state_dict = load_file(saved_model_path, device="cpu")
    print(f"Loaded state dict from {saved_model_path}")
except ImportError:
    print("Safetensors not found, trying pytorch_model.bin")
    saved_model_path = f"{lora_output_dir}/pytorch_model.bin"
    saved_state_dict = torch.load(saved_model_path, map_location='cpu')
    print(f"Loaded state dict from {saved_model_path}")

load_result = model_for_inference.load_state_dict(saved_state_dict, strict=False)
print(f"Model state dict loaded successfully: {load_result}")

loaded_model = model_for_inference
loaded_model.eval()
print("Model with fine-tuned LoRA weights successfully re-assembled and loaded for inference.")

Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

Loaded state dict from ./qwen_codealpaca_lora_finetuned/lora_adapters/model.safetensors
Model state dict loaded successfully: _IncompatibleKeys(missing_keys=['lm_head.weight'], unexpected_keys=[])
Model with fine-tuned LoRA weights successfully re-assembled and loaded for inference.


Model Merge

In [75]:
def merge_lora_modules(model: nn.Module):
    for name, module in list(model.named_modules()):
        if isinstance(module, LoRALinear):
            target_device = module.base.weight.data.device
            A_weight_on_device = module.A.weight.data.to(target_device)
            B_weight_on_device = module.B.weight.data.to(target_device)
            delta_weight = torch.matmul(B_weight_on_device, A_weight_on_device) * module.scaling
            module.base.weight.data += delta_weight
            if '.' not in name:
                parent = model
                child_name = name
            else:
                parent_name, child_name = name.rsplit('.', 1)
                parent = model.get_submodule(parent_name)

            setattr(parent, child_name, module.base)
            print(f"Merged LoRA layer: {name} successfully.")

In [76]:
merged_model = loaded_model
merge_lora_modules(merged_model)

Merged LoRA layer: model.layers.0.self_attn.q_proj successfully.
Merged LoRA layer: model.layers.0.self_attn.v_proj successfully.
Merged LoRA layer: model.layers.1.self_attn.q_proj successfully.
Merged LoRA layer: model.layers.1.self_attn.v_proj successfully.
Merged LoRA layer: model.layers.2.self_attn.q_proj successfully.
Merged LoRA layer: model.layers.2.self_attn.v_proj successfully.
Merged LoRA layer: model.layers.3.self_attn.q_proj successfully.
Merged LoRA layer: model.layers.3.self_attn.v_proj successfully.
Merged LoRA layer: model.layers.4.self_attn.q_proj successfully.
Merged LoRA layer: model.layers.4.self_attn.v_proj successfully.
Merged LoRA layer: model.layers.5.self_attn.q_proj successfully.
Merged LoRA layer: model.layers.5.self_attn.v_proj successfully.
Merged LoRA layer: model.layers.6.self_attn.q_proj successfully.
Merged LoRA layer: model.layers.6.self_attn.v_proj successfully.
Merged LoRA layer: model.layers.7.self_attn.q_proj successfully.
Merged LoRA layer: model.

In [77]:
merged_model


Qwen2ForCausalLM(
  (model): Qwen2Model(
    (embed_tokens): Embedding(151936, 896)
    (layers): ModuleList(
      (0-23): 24 x Qwen2DecoderLayer(
        (self_attn): Qwen2Attention(
          (q_proj): Linear(in_features=896, out_features=896, bias=True)
          (k_proj): Linear(in_features=896, out_features=128, bias=True)
          (v_proj): Linear(in_features=896, out_features=128, bias=True)
          (o_proj): Linear(in_features=896, out_features=896, bias=False)
        )
        (mlp): Qwen2MLP(
          (gate_proj): Linear(in_features=896, out_features=4864, bias=False)
          (up_proj): Linear(in_features=896, out_features=4864, bias=False)
          (down_proj): Linear(in_features=4864, out_features=896, bias=False)
          (act_fn): SiLUActivation()
        )
        (input_layernorm): Qwen2RMSNorm((896,), eps=1e-06)
        (post_attention_layernorm): Qwen2RMSNorm((896,), eps=1e-06)
      )
    )
    (norm): Qwen2RMSNorm((896,), eps=1e-06)
    (rotary_emb): Qwen2

Saving the Merged Model


In [78]:
merged_output_dir = f"{output_dir}/merged_model"
import os
os.makedirs(merged_output_dir, exist_ok=True)
merged_model.save_pretrained(merged_output_dir)
tokenizer.save_pretrained(merged_output_dir)

print(f"Merged model and tokenizer saved to: {merged_output_dir}")


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Merged model and tokenizer saved to: ./qwen_codealpaca_lora_finetuned/merged_model


Loading and Testing the Merged Model


In [80]:
print(f"Loading merged model from: {merged_output_dir}")
loaded_merged_model = AutoModelForCausalLM.from_pretrained(
    merged_output_dir,
    torch_dtype="auto",
    device_map="auto"
)
loaded_merged_model.eval()

# Generate text with the merged model
print("\n--- Generating with Merged Model (Python function) ---")
prompt_merged = "Write a Python function to check if a number is prime."
input_ids_merged = tokenizer.encode(prompt_merged, return_tensors="pt").to(loaded_merged_model.device)

output_merged = loaded_merged_model.generate(
    input_ids_merged,
    max_new_tokens=150,
    do_sample=True,
    top_p=0.9,
    temperature=0.7,
    num_return_sequences=1
)

generated_text_merged = tokenizer.decode(output_merged[0], skip_special_tokens=True)
print(f"\n{generated_text_merged}")


Loading merged model from: ./qwen_codealpaca_lora_finetuned/merged_model


Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]


--- Generating with Merged Model (Python function) ---

Write a Python function to check if a number is prime. 
num = 4
def is_prime(num):
    """Check if a given number is prime or not."""
    if num < 2:
        return False
    for i in range(2, int(num ** 0.5) + 1):
        if num % i == 0:
            return False
    return True

if __name__ == '__main__':
    num = 4
    print(is_prime(num)) # Output: False
'''Output: False'''
'''Explanation: 4 is not a prime number because it has factors other than 1 and itself (i.e., 2). ''' 

# Output: True
'''Explanation: 4 is a prime number as it only has two factors


In [81]:
# Another example with the merged model
print("\n--- Generating with Merged Model (C++ code explanation) ---")
prompt_merged_2 = "Explain the basic syntax of a 'for' loop in C++."
input_ids_merged_2 = tokenizer.encode(prompt_merged_2, return_tensors="pt").to(loaded_merged_model.device)

output_merged_2 = loaded_merged_model.generate(
    input_ids_merged_2,
    max_new_tokens=150,
    do_sample=True,
    top_p=0.9,
    temperature=0.7,
    num_return_sequences=1
)

generated_text_merged_2 = tokenizer.decode(output_merged_2[0], skip_special_tokens=True)
print(f"\n{generated_text_merged_2}")


--- Generating with Merged Model (C++ code explanation) ---

Explain the basic syntax of a 'for' loop in C++. A for loop is used to execute some code repeatedly until a certain condition is met. The syntax for an 'for' loop is: 

```
for (variable = start; condition; variable++)
{
    // some code
}
```

For example, the following statement will print "Hello World" 10 times:

```
#include <stdio.h>

int main() {
   int i;

   printf("Hello World\n");
   for (i = 0; i < 10; i++) 
       printf("Hello World\n");
   
   return 0;
}
``` 

This will output: 

```
Hello World
Hello World
Hello World
Hello World
Hello World
Hello World
Hello World
Hello
